# M3L2 E15 - Refactorizar el Caos

## Que vamos a ver

Este ejercicio replica el taller grupal del hands-on de la clase:
**Refactorizar el Caos**.

Te presentamos un script de chatbot que funciona pero tiene todos los problemas
que describe la lecture en la Seccion 3. Tu tarea es refactorizarlo usando LangChain.

## Este ejercicio es abierto

A diferencia de E00-E02, no hay TODOs guiados. Tenes que decidir:

- que componentes de LangChain usar,
- como separar la ingestion de la consulta,
- como estructurar el codigo para que sea mas mantenible.

## Este notebook necesita API key de OpenAI y FAISS


In [ ]:
import os
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")
print("API key cargada.")


## El script caotico original

Este script fue escrito rapido para una demo. Funciona, pero tiene todos los
problemas tipicos de un script tactico (Lecture - Seccion 3.4).

Ejecutalo primero para entender que hace.
Luego identifica los problemas antes de refactorizarlo.


In [ ]:
# =====================================================================
# SCRIPT CAOTICO ORIGINAL - NO MODIFICAR
# =====================================================================

from openai import OpenAI

# Datos de producto hardcodeados - en produccion vendrian de un DB
PRODUCT_DATA = """
Producto: Plan Basico
Precio: $9.99 por mes
Usuarios: hasta 5
Almacenamiento: 10GB
Soporte: Email

Producto: Plan Pro
Precio: $29.99 por mes
Usuarios: hasta 20
Almacenamiento: 100GB
Soporte: Email + Chat

Producto: Plan Enterprise
Precio: $99.99 por mes
Usuarios: ilimitados
Almacenamiento: 1TB
Soporte: Email + Chat + Telefono + SLA 24/7
"""

FAQ_DATA = """
P: Como cancelo mi suscripcion?
R: Podes cancelar en cualquier momento desde tu panel de usuario. No hay penalidades.

P: Hay prueba gratuita?
R: Si, ofrecemos 14 dias gratis sin tarjeta de credito.

P: Puedo cambiar de plan despues?
R: Si, podes hacer upgrade o downgrade en cualquier momento. El cambio aplica desde el proximo ciclo.
"""


def ask_the_chatbot(user_input, language="spanish"):
    # Mezcla de ingestion + retrieval + prompt + llamada al modelo en una sola funcion
    
    client = OpenAI()  # se crea un nuevo cliente en cada llamada
    
    # Construye el contexto concatenando TODO
    # Sin retrieval: manda todos los datos aunque no sean relevantes
    full_context = PRODUCT_DATA + "\n\n" + FAQ_DATA

    if language == "spanish":
        system = "Eres un asistente de soporte. Responde en espanol."
    elif language == "english":
        system = "You are a support assistant. Answer in English."
    else:
        system = "Eres un asistente. Responde brevemente."

    # Prompt armado como string manual con condicionales mezclados
    the_prompt = system + "\n\nInformacion disponible:\n" + full_context + "\n\nPregunta del usuario: " + user_input
    
    # Llamada hardcodeada a gpt-4o-mini, temperatura hardcodeada
    resp = OpenAI().chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": the_prompt}],
        temperature=0.7,  # temperatura elegida al azar
    )
    
    result = resp.choices[0].message.content
    return result


# Probar el script original
print("=== Chatbot original ===")
print(ask_the_chatbot("Cuanto cuesta el plan Pro?"))
print()
print(ask_the_chatbot("How do I cancel my subscription?", language="english"))


## Identificar los problemas antes de refactorizar

Antes de tocar el codigo, lista los problemas que ves en `ask_the_chatbot`.

Guia basada en la Lecture Seccion 3.4:

- Hay prompts construidos como strings? Donde?
- Hay llamadas al modelo hardcodeadas? Donde?
- Se mezclan responsabilidades en la misma funcion? Cuales?
- Hay logica de idioma mezclada con el prompt? Como se separa?
- Si quiero cambiar el modelo, cuantos lugares hay que tocar?
- Si quiero agregar retrieval real, que cambios son necesarios?


In [ ]:
# Ejercicio de analisis: completa esta lista con los problemas que identificaste

problemas_identificados = [
    # "El cliente OpenAI se crea en cada llamada en vez de una vez",
    # "...",
    # Agrega tus propios hallazgos
]

print("Problemas identificados:")
for i, p in enumerate(problemas_identificados, 1):
    print(f"  {i}. {p}")

if not problemas_identificados:
    print("  (lista vacia - completar antes de refactorizar)")


## Mapa de refactorizacion

Antes de escribir codigo, mapea que componente de LangChain reemplaza cada parte del script.

Referencia de componentes (Lecture - Seccion 8):

| Componente | Para que sirve |
|---|---|
| `ChatOpenAI` | Encapsula el modelo |
| `ChatPromptTemplate` | Estructurar el prompt |
| `StrOutputParser` | Extraer texto de la respuesta |
| `FAISS.from_texts()` | Crear vector store en memoria |
| `vectorstore.as_retriever()` | Encapsular busqueda |
| `LCEL (\|)` | Conectar componentes |
| `RunnablePassthrough` | Pasar inputs sin modificar |


In [ ]:
# Mapa de refactorizacion: completa este diccionario
# Mapea cada problema del script al componente de LangChain que lo resuelve

mapa_refactorizacion = {
    "Llamada hardcodeada a OpenAI": "",            # que componente usarias?
    "Prompt como string manual": "",               # que componente?
    "Todo el contexto sin retrieval": "",          # como separias ingestion y consulta?
    "Logica de idioma mezclada en el prompt": "",  # como lo separias?
    "Temperatura hardcodeada": "",                 # donde la centralizas?
}

print("Mapa de refactorizacion:")
for problema, solucion in mapa_refactorizacion.items():
    estado = solucion if solucion else "(pendiente)"
    print(f"  {problema} → {estado}")


## Tu refactorizacion

Ahora implementa el chatbot usando LangChain.

No hay TODOs guiados. Usa lo aprendido en E00, E01 y E02.

Criterios de exito:

- El modelo esta configurado en un solo lugar.
- El prompt es un `ChatPromptTemplate`.
- El flujo de datos es visible (usando LCEL).
- La ingestion (crear vector store) esta separada de la consulta (invocar chain).
- El retriever busca solo los documentos relevantes (no manda todo el contexto).
- Se puede cambiar el modelo cambiando solo una linea.


In [ ]:
# =====================================================================
# TU REFACTORIZACION - espacio libre para implementar
# =====================================================================

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Datos de la empresa (mismos que el script original, separados por documento)
DOCUMENTOS = [
    "Plan Basico: $9.99 por mes. Hasta 5 usuarios. 10GB almacenamiento. Soporte por email.",
    "Plan Pro: $29.99 por mes. Hasta 20 usuarios. 100GB almacenamiento. Soporte email y chat.",
    "Plan Enterprise: $99.99 por mes. Usuarios ilimitados. 1TB almacenamiento. Soporte completo con SLA 24/7.",
    "Cancelacion: se puede cancelar en cualquier momento desde el panel. Sin penalidades.",
    "Prueba gratuita: 14 dias gratis sin tarjeta de credito.",
    "Cambio de plan: se puede hacer upgrade o downgrade en cualquier momento. Aplica desde el proximo ciclo.",
]

# Tu implementacion aqui:

# INGESTION
# ...

# CHAIN DE CONSULTA
# ...

# FUNCION REFACTORIZADA
def ask_chatbot_refactored(question: str) -> str:
    """Chatbot refactorizado con LangChain."""
    # ...
    pass


# Probar la version refactorizada
print("=== Chatbot refactorizado ===")
respuesta = ask_chatbot_refactored("Cuanto cuesta el plan Pro?")
print(respuesta)


## Verificacion: comparar las dos versiones

Ejecuta las dos versiones con las mismas preguntas y compara:
- las respuestas (deben ser similares),
- el codigo (debe ser mas legible),
- la capacidad de debugging (debe ser mayor).


In [ ]:
preguntas = [
    "Cuanto cuesta el plan Pro?",
    "Hay prueba gratuita?",
    "Cuantos usuarios soporta el Plan Basico?",
]

print("Comparacion de respuestas:")
print()

for pregunta in preguntas:
    print(f"Pregunta: {pregunta}")
    
    try:
        r_original = ask_the_chatbot(pregunta)
        print(f"  Original  : {r_original[:80]}...")
    except Exception as e:
        print(f"  Original  : ERROR - {e}")
    
    try:
        r_refactored = ask_chatbot_refactored(pregunta)
        print(f"  Refactored: {r_refactored[:80] if r_refactored else 'None - TODO no completado'}...")
    except Exception as e:
        print(f"  Refactored: ERROR - {e}")
    
    print()


## Reflexion final

Responde estas preguntas despues de completar la refactorizacion:

1. Cuantos componentes independientes tiene tu version refactorizada?
2. Si el cliente quiere cambiar de `gpt-4o-mini` a `gpt-4o`, cuantas lineas hay que tocar?
3. Si quieren agregar documentos nuevos, como lo harian en la version original vs la tuya?
4. Si la respuesta es mala, como debugging cada paso en tu version?
5. La lecture menciona que `Chain` se usa para flujos fijos y `Agent` para flujos dinamicos.
   Este chatbot necesita un Agent o alcanza con una Chain? Por que?


In [ ]:
# Espacio para tus respuestas (como comentarios)

# 1. Componentes independientes: ...
# 2. Para cambiar el modelo: ...
# 3. Para agregar documentos: ...
# 4. Para debuggear: ...
# 5. Chain vs Agent: ...

print("Reflexion completada.")


## Checklist de produccion (Lecture - Seccion 22)

Usa el checklist de la lecture para evaluar tu refactorizacion.


In [ ]:
checklist = {
    # Arquitectura
    "Separaste ingestion y consulta": None,
    "El retriever esta encapsulado": None,
    "El prompt usa ChatPromptTemplate": None,
    "El modelo esta configurado centralmente": None,
    "El flujo esta definido como Chain o LCEL": None,
    # Configuracion
    "No hay API keys hardcodeadas": None,
    "El modelo es configurable (no hardcodeado)": None,
    # Debugging
    "Se pueden inspeccionar los documentos recuperados": None,
    "Se puede ver el prompt final": None,
}

print("=== Checklist de produccion ===")
aprobados = 0
for item, estado in checklist.items():
    if estado is True:
        simbolo = "[OK]"
        aprobados += 1
    elif estado is False:
        simbolo = "[NO]"
    else:
        simbolo = "[--]"
    print(f"  {simbolo} {item}")

print(f"\nAprobados: {aprobados}/{len(checklist)}")
